# Train weighted tokens on all exported training pages
Batch size 1 = one annotated page. Same LoRA and selectable loss configurations as 1pageTest. No validation samples enter the loader.

Load only manifest-approved training exports.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "project.json").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pipeline_helpers as P
import json
from PIL import Image, ImageDraw, ImageFont
ROOT = P.ROOT
MODELS = P.read_json(ROOT / "models.json")
SPLITS = ["train", "val"]  # Or ["train"] / ["val"].
FORCE_RERUN = False
items = P.pages(SPLITS)
print("Pages:", len(items))
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
from weighted_tokens_helpers import load_joint_sample, build_masked_target, collate_weighted_tokens, weighted_supervised_loss
SAMPLE_PATHS = P.exported_samples("train")
samples = [load_joint_sample(path) for path in SAMPLE_PATHS]
print("Training samples:", len(samples))
print("Not yet exported:", len(P.pages(("train",))) - len(samples))


Choose weights and run settings. Epochs remain configurable; default matches the earlier 100-epoch comparison.

In [ ]:
# Select one named loss configuration. The old weights remain available.
WEIGHT_CONFIGS = {
    "1stWeightConfig": {
        "decision_weight": 1.0,
        "formatting_weight": 0.1,
        "end_weight": 0.1,
        "field_weight": 0.0,
        "ignore_index": -100,
        "train_decisions": ["P IDs", "A IDs", "B IDs", "UNKNOWN"],
    },
    "2ndWeightConfig": {
        "decision_weight": 1.0,
        "formatting_weight": 1.0,
        "end_weight": 1.0,
        "field_weight": 1.0,
        "ignore_index": -100,
        "train_decisions": ["P IDs", "A IDs", "B IDs", "UNKNOWN"],
    },
}
RUN_NAMES = {
    "1stWeightConfig": "weighted_tokens_structure_all_train_01",
    "2ndWeightConfig": "weighted_tokens_full_assistant_all_train_01",
}
WEIGHT_CONFIG_NAME = "2ndWeightConfig"
LOSS_CONFIG = WEIGHT_CONFIGS[WEIGHT_CONFIG_NAME].copy()

# Model, LoRA, data and training settings match the previous 100-epoch run.
CONFIG = H.experiment(
    name=RUN_NAMES[WEIGHT_CONFIG_NAME],
    decoder="attention",  # q_proj, k_proj, v_proj, o_proj
    last_layers=8,
    projector=False,
    vision=False,
    rank=4,
    epochs=100,
    lr=1e-4,
    max_tiles=1,
    accumulation=1,
)
CONFIG["view"] = "annotated"
CONFIG["max_length"] = 4096
CONFIG["loss"] = LOSS_CONFIG
CONFIG["weight_config_name"] = WEIGHT_CONFIG_NAME
RUN = ROOT / "data/training/runs" / CONFIG["name"]
print("Selected weights:", WEIGHT_CONFIG_NAME)
print(json.dumps(CONFIG, indent=2))

CONFIG["split_manifest_sha256"] = P.file_hash(P.MANIFEST)
CONFIG["train_sequences"] = P.split_manifest()["sequences"]["train"]


Preview the first target without loading the VL model.

In [ ]:
from transformers import AutoTokenizer

offset_tokenizer = AutoTokenizer.from_pretrained(
    str(H.MODEL_DIR), trust_remote_code=True,
    local_files_only=True, use_fast=True,
)
preview = build_masked_target(offset_tokenizer, samples[0]["target"], separator="<|im_end|>", loss_config=LOSS_CONFIG)
print("Assistant JSON:", preview["text"])
print("Decision spans:", [preview["text"][start:end] for start, end in preview["spans"]])
print("\nToken                     Weight")
for weight, (start, end) in zip(preview["token_weights"], preview["offsets"]):
    piece = (preview["text"] + "<|im_end|>")[start:end]
    print(f"{piece!r:25} {weight:.2f}")


Load the local CUDA model and attach attention LoRA.

In [ ]:
import random
import torch

if RUN.exists():
    raise FileExistsError(f"Choose a new run name; {RUN} already exists.")
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
H.cleanup()
model, tokenizer = H.load_base()
parts = H.attach_lora(model, CONFIG)
print(parts)
assert not any(p.requires_grad for p in model.vision_model.parameters())
assert not any(p.requires_grad for p in model.mlp1.parameters())

def collator(items):
    return collate_weighted_tokens(items, model, tokenizer, offset_tokenizer, CONFIG)

batch = collator([samples[0]])
assert (batch["labels"][:, :batch["answer_start"]] == -100).all()
print("Total tokens:", batch["input_ids"].numel())
assert (batch["token_weights"][:, :batch["answer_start"]] == 0).all()
print("Total loss weight:", float(batch["token_weights"].sum()))
print("All supervised tokens:", int((batch["token_weights"] > 0).sum()))
del batch


Train across all training samples each epoch, then save adapters.

In [ ]:
from torch.utils.data import DataLoader

if LOSS_CONFIG["ignore_index"] != -100:
    raise ValueError("Our label mask uses ignore_index=-100.")
if RUN.exists():
    raise FileExistsError("Choose a new run name before training.")
RUN.mkdir(parents=True)
H.write_json(RUN / "config.json", CONFIG)
H.write_json(RUN / "split_manifest.json", P.split_manifest())
H.write_json(RUN / "sample_paths.json", [str(path) for path in SAMPLE_PATHS])
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=CONFIG["lr"])
loader = DataLoader(samples, batch_size=1, shuffle=True, collate_fn=collator, num_workers=0)
history = []
first_update = True
accumulation = CONFIG["accumulation"]
H.training_mode(model)

for epoch in range(CONFIG["epochs"]):
    optimizer.zero_grad(set_to_none=True)
    epoch_loss = 0.0
    for step, batch in enumerate(loader):
        loss = weighted_supervised_loss(model, batch)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss. Stop and inspect the sample/model.")
        epoch_loss += float(loss.detach())
        # Normalize a final incomplete accumulation group correctly.
        group_start = (step // accumulation) * accumulation
        group_size = min(accumulation, len(loader) - group_start)
        (loss / group_size).backward()
        if (step + 1) % accumulation == 0 or step + 1 == len(loader):
            if first_update:
                report = H.grad_report(model)
                print("First-update gradients:", report)
                if report.get("language_model", 0.0) <= 0:
                    raise RuntimeError("No language LoRA gradient.")
                first_update = False
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        del batch, loss
    row = {"epoch": epoch + 1, "weighted_token_loss": epoch_loss / len(loader)}
    history.append(row)
    print(row)
    H.write_json(RUN / "loss_history.json", history)



model.eval()
H.save_adapters(model, RUN, CONFIG)
print("Saved adapters:", RUN)


## View saved results
These cells only read saved results. They do not run models or change cached files. Training shows targets and run-wide loss history; validation shows saved predictions.

In [ ]:
from pathlib import Path
import sys, importlib
view_root = Path.cwd().resolve()
if not (view_root / "project.json").is_file():
    view_root = view_root.parent
sys.path.insert(0, str(view_root))
import result_viewer as V
importlib.reload(V)
VIEW_STAGE = 'training'
VIEW_SPLIT = "train"  # Or "val".
VIEW_RUN = Path(RUN) if "RUN" in globals() else view_root / "data/training/runs/weighted_tokens_full_assistant_all_train_01"


### 1. View a random page
Rerun this cell to choose another page with a saved result in this split.

In [ ]:
viewed_page = V.show_random(VIEW_STAGE, split=VIEW_SPLIT, run_folder=VIEW_RUN)


### 2. View all pages of a sequence
Enter a sequence ID, or leave it empty to use the random page’s sequence. Missing stage results are reported for each page.

In [ ]:
VIEW_SEQUENCE_ID = ""  # Example: "seq_71033fec4de494fc"
sequence_to_view = VIEW_SEQUENCE_ID or (viewed_page["sequence_id"] if viewed_page else "")
if sequence_to_view:
    V.show_sequence(sequence_to_view, VIEW_STAGE, split=VIEW_SPLIT, run_folder=VIEW_RUN)
else:
    print("Set VIEW_SEQUENCE_ID to a sequence in VIEW_SPLIT.")
